
# Grain Robustness Q1 — Notebook 03
## Particiones estratificadas, agrupadas y congeladas

Este notebook utiliza exclusivamente:

`NOTEBOOK02_REDUNDANCY_REPRESENTATIONS_OUTPUTS.zip`

### Objetivo

Crear una partición reproducible y sin fuga de información para cada dataset:

- **60 % entrenamiento**
- **20 % calibración**
- **20 % prueba final**

### Principios metodológicos

1. Los registros con el mismo `duplicate_group_id` permanecen juntos.
2. Las clases se estratifican mediante `StratifiedGroupKFold`.
3. La selección de los folds se basa solo en tamaños y etiquetas, nunca en el
   desempeño de modelos.
4. El conjunto de calibración se reservará para calibración probabilística y
   predicción conformal.
5. La selección de hiperparámetros se hará exclusivamente mediante validación
   cruzada dentro del conjunto de entrenamiento.
6. El conjunto de prueba queda congelado y no debe utilizarse para seleccionar
   representaciones, modelos ni hiperparámetros.
7. Las cuatro representaciones comparten exactamente los mismos registros y
   asignaciones de partición.
8. Se utiliza **CPU**; no se requiere GPU.


## 1. Preparar Colab

In [ ]:
!pip -q install pyarrow openpyxl


In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)


## 2. Importaciones y configuración

In [ ]:

from pathlib import Path
from datetime import datetime, timezone
from itertools import permutations
import hashlib
import json
import os
import platform
import shutil
import sys
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold

RANDOM_SEED = 20260803
N_SPLITS = 5

# Semillas futuras para modelos estocásticos. No modifican la partición.
MODEL_SEEDS = [17, 42, 73, 101, 2026]

LOCAL_ROOT = Path('/content/Grain_Robustness_Q1_notebook03')
INPUT_DIR = LOCAL_ROOT / '00_input'
EXTRACT_DIR = LOCAL_ROOT / '01_extracted_notebook02'
OUTPUT_DIR = LOCAL_ROOT / '02_outputs'
SPLIT_DIR = OUTPUT_DIR / '05_frozen_splits'
PROTOCOL_DIR = OUTPUT_DIR / '03_protocol'
RESULTS_DIR = OUTPUT_DIR / '07_results' / '03_FROZEN_SPLITS'
FIGURES_DIR = OUTPUT_DIR / '08_figures' / '03_FROZEN_SPLITS'
TABLES_DIR = OUTPUT_DIR / '09_tables' / '03_FROZEN_SPLITS'

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/Grain_Robustness_Q1')

for directory in [
    INPUT_DIR,
    EXTRACT_DIR,
    SPLIT_DIR,
    PROTOCOL_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

DATASETS = ['dry_bean', 'iniap_rice']
REPRESENTATIONS = [
    'R1_full_native',
    'R2_measured_core',
    'R3_invariant_core',
    'R4_hybrid_log_area',
]

print('Espacio local:', LOCAL_ROOT)
print('Destino en Drive:', DRIVE_PROJECT_ROOT)
print('CPU disponibles:', os.cpu_count())
print('Python:', sys.version.split()[0])


## 3. Localizar el ZIP del Notebook 02

In [ ]:

from google.colab import files

candidate_paths = [
    Path('/content/NOTEBOOK02_REDUNDANCY_REPRESENTATIONS_OUTPUTS.zip'),
    (
        DRIVE_PROJECT_ROOT / '07_results'
        / 'NOTEBOOK02_REDUNDANCY_REPRESENTATIONS_OUTPUTS.zip'
    ),
    (
        DRIVE_PROJECT_ROOT
        / 'NOTEBOOK02_REDUNDANCY_REPRESENTATIONS_OUTPUTS.zip'
    ),
]

zip_source = next(
    (path for path in candidate_paths if path.exists()),
    None,
)

if zip_source is None:
    print(
        'No se encontró automáticamente el ZIP. '
        'Selecciona NOTEBOOK02_REDUNDANCY_REPRESENTATIONS_OUTPUTS.zip.'
    )
    uploaded = files.upload()
    uploaded_names = list(uploaded.keys())

    if len(uploaded_names) != 1:
        raise RuntimeError(
            'Debes cargar exactamente un archivo ZIP.'
        )

    zip_source = Path('/content') / uploaded_names[0]

if zip_source.suffix.lower() != '.zip':
    raise ValueError(f'El archivo no es ZIP: {zip_source}')

zip_local = (
    INPUT_DIR
    / 'NOTEBOOK02_REDUNDANCY_REPRESENTATIONS_OUTPUTS.zip'
)
shutil.copy2(zip_source, zip_local)

print('ZIP utilizado:', zip_source)
print('Copia local:', zip_local)
print('Tamaño MB:', round(zip_local.stat().st_size / 1_000_000, 3))


## 4. Extraer y validar las representaciones

In [ ]:

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_local, 'r') as archive:
    archive.extractall(EXTRACT_DIR)

validation_path = (
    EXTRACT_DIR / '07_results'
    / '02_REDUNDANCY_REPRESENTATIONS'
    / '02_final_representation_validation.csv'
)
catalog_path = (
    EXTRACT_DIR / '03_protocol'
    / 'representation_catalog.csv'
)
representation_protocol_path = (
    EXTRACT_DIR / '03_protocol'
    / 'representation_protocol_v1.json'
)

required = [
    validation_path,
    catalog_path,
    representation_protocol_path,
]

for dataset_id in DATASETS:
    for representation in REPRESENTATIONS:
        required.append(
            EXTRACT_DIR / '04_processed_features'
            / dataset_id
            / f'{representation}.parquet'
        )

missing = [str(path) for path in required if not path.exists()]

if missing:
    raise FileNotFoundError(
        'Faltan archivos requeridos:\n'
        + '\n'.join(missing)
    )

previous_validation = pd.read_csv(validation_path)
boolean_columns = [
    'file_exists',
    'rows_ok',
    'record_ids_unique',
    'group_ids_complete',
    'class_labels_complete',
    'feature_count_ok',
]

if not previous_validation[boolean_columns].all(axis=None):
    raise RuntimeError(
        'La salida del Notebook 02 no superó todas las validaciones.'
    )

if previous_validation['missing_feature_cells'].sum() != 0:
    raise RuntimeError(
        'Las representaciones contienen valores faltantes.'
    )

print('Salida del Notebook 02 validada correctamente.')
display(previous_validation)


## 5. Cargar y alinear las cuatro representaciones

In [ ]:

representations = {}

for dataset_id in DATASETS:
    representations[dataset_id] = {}

    reference_keys = None

    for representation in REPRESENTATIONS:
        path = (
            EXTRACT_DIR / '04_processed_features'
            / dataset_id
            / f'{representation}.parquet'
        )
        dataframe = pd.read_parquet(path)
        dataframe = dataframe.sort_values('record_id').reset_index(drop=True)

        key_columns = [
            'record_id',
            'duplicate_group_id',
            'class_label',
        ]
        keys = dataframe[key_columns].copy()

        if reference_keys is None:
            reference_keys = keys
        elif not reference_keys.equals(keys):
            raise RuntimeError(
                f'Las representaciones no están alineadas en '
                f'{dataset_id}: {representation}'
            )

        representations[dataset_id][representation] = dataframe

    print(
        dataset_id,
        len(reference_keys),
        'registros alineados en 4 representaciones.'
    )


## 6. Crear cinco folds estratificados y agrupados

In [ ]:

def create_grouped_folds(reference_dataframe):
    y = reference_dataframe['class_label'].astype(str).to_numpy()
    groups = reference_dataframe[
        'duplicate_group_id'
    ].astype(str).to_numpy()

    splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_SEED,
    )

    fold_assignment = np.full(
        len(reference_dataframe),
        fill_value=-1,
        dtype=int,
    )

    # X no participa en la división; se suministra una columna ficticia.
    dummy_x = np.zeros((len(reference_dataframe), 1))

    for fold_id, (_, fold_indices) in enumerate(
        splitter.split(dummy_x, y, groups)
    ):
        fold_assignment[fold_indices] = fold_id

    if np.any(fold_assignment < 0):
        raise RuntimeError(
            'Al menos un registro quedó sin fold.'
        )

    return fold_assignment


fold_assignments = {}

for dataset_id in DATASETS:
    reference = representations[dataset_id][
        'R2_measured_core'
    ]

    folds = create_grouped_folds(reference)
    fold_assignments[dataset_id] = folds

    print(
        dataset_id,
        pd.Series(folds).value_counts().sort_index().to_dict(),
    )



## 7. Elegir folds de calibración y prueba sin observar modelos

De los cinco folds:

- uno se asigna a calibración;
- uno se asigna a prueba;
- los tres restantes se asignan a entrenamiento.

Se evalúan las 20 parejas ordenadas posibles. La función de costo considera:

1. desviación del tamaño respecto de 60/20/20;
2. desviación de las proporciones de clase respecto del dataset completo;
3. desviación del tamaño de cada clase respecto de 60/20/20.

La elección utiliza únicamente identificadores y etiquetas.


In [ ]:

def assignment_cost(dataframe, folds, calibration_fold, test_fold):
    partition = np.full(
        len(dataframe),
        'train',
        dtype=object,
    )
    partition[folds == calibration_fold] = 'calibration'
    partition[folds == test_fold] = 'test'

    expected_proportions = {
        'train': 0.60,
        'calibration': 0.20,
        'test': 0.20,
    }

    total_n = len(dataframe)
    labels = dataframe['class_label'].astype(str)
    overall_class_proportions = (
        labels.value_counts(normalize=True)
    )

    size_penalty = 0.0
    class_proportion_penalty = 0.0
    class_size_penalty = 0.0

    for partition_name, expected_proportion in (
        expected_proportions.items()
    ):
        mask = partition == partition_name
        observed_size_proportion = mask.mean()

        size_penalty += abs(
            observed_size_proportion - expected_proportion
        )

        observed_class_proportions = (
            labels[mask]
            .value_counts(normalize=True)
            .reindex(
                overall_class_proportions.index,
                fill_value=0.0,
            )
        )

        class_proportion_penalty += (
            observed_class_proportions
            - overall_class_proportions
        ).abs().mean()

        for class_label, class_total in (
            labels.value_counts().items()
        ):
            observed_class_count = int(
                ((labels == class_label) & mask).sum()
            )
            expected_class_count = (
                expected_proportion * class_total
            )
            class_size_penalty += (
                abs(observed_class_count - expected_class_count)
                / max(class_total, 1)
            )

    total_cost = (
        3.0 * size_penalty
        + 2.0 * class_proportion_penalty
        + class_size_penalty
    )

    return {
        'calibration_fold': calibration_fold,
        'test_fold': test_fold,
        'size_penalty': size_penalty,
        'class_proportion_penalty': class_proportion_penalty,
        'class_size_penalty': class_size_penalty,
        'total_cost': total_cost,
    }


selection_rows = []
selected_fold_pairs = {}

for dataset_id in DATASETS:
    reference = representations[dataset_id][
        'R2_measured_core'
    ]
    folds = fold_assignments[dataset_id]

    dataset_candidates = []

    for calibration_fold, test_fold in permutations(
        range(N_SPLITS),
        2,
    ):
        result = assignment_cost(
            reference,
            folds,
            calibration_fold,
            test_fold,
        )
        result['dataset_id'] = dataset_id
        dataset_candidates.append(result)
        selection_rows.append(result)

    candidates = pd.DataFrame(dataset_candidates)
    best = candidates.sort_values(
        [
            'total_cost',
            'test_fold',
            'calibration_fold',
        ]
    ).iloc[0]

    selected_fold_pairs[dataset_id] = {
        'calibration_fold': int(best['calibration_fold']),
        'test_fold': int(best['test_fold']),
    }

fold_selection = pd.DataFrame(selection_rows)
fold_selection.to_csv(
    RESULTS_DIR / '01_fold_pair_selection_candidates.csv',
    index=False,
    encoding='utf-8-sig',
)

print('Folds seleccionados:')
print(json.dumps(selected_fold_pairs, indent=2))


## 8. Construir y validar los manifiestos de partición

In [ ]:

split_manifests = {}
split_summary_rows = []
class_summary_rows = []
leakage_rows = []

for dataset_id in DATASETS:
    reference = representations[dataset_id][
        'R2_measured_core'
    ].copy()

    folds = fold_assignments[dataset_id]
    selected = selected_fold_pairs[dataset_id]

    partition = np.full(
        len(reference),
        'train',
        dtype=object,
    )
    partition[
        folds == selected['calibration_fold']
    ] = 'calibration'
    partition[
        folds == selected['test_fold']
    ] = 'test'

    manifest = reference[
        [
            'dataset_id',
            'record_id',
            'source_excel_row',
            'feature_fingerprint_sha256',
            'full_row_fingerprint_sha256',
            'duplicate_group_id',
            'exact_duplicate_count',
            'class_label',
        ]
    ].copy()

    manifest['fold_id'] = folds
    manifest['partition'] = partition
    manifest['split_seed'] = RANDOM_SEED
    manifest['split_version'] = 'grouped_stratified_60_20_20_v1'

    split_manifests[dataset_id] = manifest

    manifest.to_csv(
        SPLIT_DIR / f'{dataset_id}_split_manifest_v1.csv',
        index=False,
        encoding='utf-8-sig',
    )
    manifest.to_parquet(
        SPLIT_DIR / f'{dataset_id}_split_manifest_v1.parquet',
        index=False,
    )

    for partition_name in [
        'train',
        'calibration',
        'test',
    ]:
        part = manifest[
            manifest['partition'] == partition_name
        ]

        split_summary_rows.append({
            'dataset_id': dataset_id,
            'partition': partition_name,
            'n_records': len(part),
            'proportion': len(part) / len(manifest),
            'n_groups': part['duplicate_group_id'].nunique(),
            'n_classes': part['class_label'].nunique(),
        })

        class_counts = (
            part['class_label']
            .value_counts()
            .sort_index()
        )

        for class_label, n_records in class_counts.items():
            class_summary_rows.append({
                'dataset_id': dataset_id,
                'partition': partition_name,
                'class_label': class_label,
                'n_records': int(n_records),
                'within_partition_proportion': (
                    n_records / len(part)
                ),
            })

    # Cada grupo debe pertenecer a una sola partición.
    group_partition_counts = (
        manifest.groupby('duplicate_group_id')[
            'partition'
        ].nunique()
    )

    # Cada registro debe aparecer exactamente una vez.
    record_counts = manifest['record_id'].value_counts()

    # Las particiones no deben compartir huellas completas.
    partition_fingerprints = {
        name: set(
            manifest.loc[
                manifest['partition'] == name,
                'full_row_fingerprint_sha256',
            ]
        )
        for name in ['train', 'calibration', 'test']
    }

    leakage_rows.append({
        'dataset_id': dataset_id,
        'groups_in_multiple_partitions': int(
            (group_partition_counts > 1).sum()
        ),
        'duplicate_record_ids': int(
            (record_counts > 1).sum()
        ),
        'train_calibration_fingerprint_overlap': len(
            partition_fingerprints['train']
            & partition_fingerprints['calibration']
        ),
        'train_test_fingerprint_overlap': len(
            partition_fingerprints['train']
            & partition_fingerprints['test']
        ),
        'calibration_test_fingerprint_overlap': len(
            partition_fingerprints['calibration']
            & partition_fingerprints['test']
        ),
    })

split_summary = pd.DataFrame(split_summary_rows)
class_summary = pd.DataFrame(class_summary_rows)
leakage_audit = pd.DataFrame(leakage_rows)

split_summary.to_csv(
    TABLES_DIR / 'Table_split_summary.csv',
    index=False,
    encoding='utf-8-sig',
)
class_summary.to_csv(
    TABLES_DIR / 'Table_split_class_distribution.csv',
    index=False,
    encoding='utf-8-sig',
)
leakage_audit.to_csv(
    RESULTS_DIR / '02_split_leakage_audit.csv',
    index=False,
    encoding='utf-8-sig',
)

numeric_leakage_columns = [
    column
    for column in leakage_audit.columns
    if column != 'dataset_id'
]

if leakage_audit[numeric_leakage_columns].to_numpy().sum() != 0:
    raise RuntimeError(
        'Se detectó fuga entre particiones.'
    )

display(split_summary)
display(class_summary)
display(leakage_audit)


## 9. Añadir la partición a cada representación

In [ ]:

enriched_manifest_rows = []

for dataset_id in DATASETS:
    manifest = split_manifests[dataset_id][
        ['record_id', 'fold_id', 'partition']
    ]

    dataset_output_dir = SPLIT_DIR / dataset_id
    dataset_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for representation in REPRESENTATIONS:
        source = representations[dataset_id][
            representation
        ].copy()

        enriched = source.merge(
            manifest,
            on='record_id',
            how='left',
            validate='one_to_one',
        )

        if enriched['partition'].isna().any():
            raise RuntimeError(
                f'Faltan asignaciones para {dataset_id}, '
                f'{representation}.'
            )

        enriched = enriched.sort_values(
            'record_id'
        ).reset_index(drop=True)

        csv_path = (
            dataset_output_dir
            / f'{representation}_with_split.csv'
        )
        parquet_path = (
            dataset_output_dir
            / f'{representation}_with_split.parquet'
        )

        enriched.to_csv(
            csv_path,
            index=False,
            encoding='utf-8-sig',
        )
        enriched.to_parquet(
            parquet_path,
            index=False,
        )

        for path, file_format in [
            (csv_path, 'csv'),
            (parquet_path, 'parquet'),
        ]:
            enriched_manifest_rows.append({
                'dataset_id': dataset_id,
                'representation': representation,
                'format': file_format,
                'path': str(path),
                'rows': len(enriched),
                'size_bytes': path.stat().st_size,
            })

enriched_file_manifest = pd.DataFrame(
    enriched_manifest_rows
)

enriched_file_manifest.to_csv(
    RESULTS_DIR / '03_split_enriched_file_manifest.csv',
    index=False,
    encoding='utf-8-sig',
)

display(enriched_file_manifest)


## 10. Comprobar alineación entre representaciones después de la partición

In [ ]:

alignment_rows = []

for dataset_id in DATASETS:
    reference_path = (
        SPLIT_DIR / dataset_id
        / 'R1_full_native_with_split.parquet'
    )
    reference = (
        pd.read_parquet(reference_path)[
            ['record_id', 'class_label', 'partition', 'fold_id']
        ]
        .sort_values('record_id')
        .reset_index(drop=True)
    )

    for representation in REPRESENTATIONS:
        path = (
            SPLIT_DIR / dataset_id
            / f'{representation}_with_split.parquet'
        )
        observed = (
            pd.read_parquet(path)[
                ['record_id', 'class_label', 'partition', 'fold_id']
            ]
            .sort_values('record_id')
            .reset_index(drop=True)
        )

        alignment_rows.append({
            'dataset_id': dataset_id,
            'representation': representation,
            'rows': len(observed),
            'alignment_exact': reference.equals(observed),
        })

alignment_audit = pd.DataFrame(alignment_rows)
alignment_audit.to_csv(
    RESULTS_DIR / '04_representation_split_alignment.csv',
    index=False,
    encoding='utf-8-sig',
)

if not alignment_audit['alignment_exact'].all():
    raise RuntimeError(
        'Las representaciones no comparten exactamente la misma partición.'
    )

print('Alineación de particiones: CORRECTA')
display(alignment_audit)


## 11. Figuras de distribución por partición

In [ ]:

for dataset_id in DATASETS:
    plot_data = class_summary[
        class_summary['dataset_id'] == dataset_id
    ].copy()

    classes = sorted(
        plot_data['class_label'].unique()
    )
    partitions = ['train', 'calibration', 'test']

    x = np.arange(len(classes))
    width = 0.25

    figure, axis = plt.subplots(figsize=(11, 6))

    for partition_index, partition_name in enumerate(partitions):
        values = (
            plot_data[
                plot_data['partition'] == partition_name
            ]
            .set_index('class_label')['n_records']
            .reindex(classes, fill_value=0)
            .to_numpy()
        )

        axis.bar(
            x + (partition_index - 1) * width,
            values,
            width=width,
            label=partition_name,
        )

    axis.set_xticks(x)
    axis.set_xticklabels(
        classes,
        rotation=35,
        ha='right',
    )
    axis.set_xlabel('Clase')
    axis.set_ylabel('Número de registros')
    axis.set_title(
        f'Distribución de clases por partición — {dataset_id}'
    )
    axis.legend()
    axis.grid(True, axis='y', alpha=0.3)
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR / f'split_class_distribution_{dataset_id}.png',
        dpi=300,
        bbox_inches='tight',
    )
    figure.savefig(
        FIGURES_DIR / f'split_class_distribution_{dataset_id}.pdf',
        bbox_inches='tight',
    )
    plt.show()


## 12. Congelar el protocolo de modelado

In [ ]:

modeling_protocol = {
    'protocol_name': 'Grain_Robustness_Q1_frozen_split_protocol_v1',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'split_seed': RANDOM_SEED,
    'model_seeds': MODEL_SEEDS,
    'split_method': (
        'StratifiedGroupKFold with five folds; one fold for calibration, '
        'one fold for final testing, and three folds for training.'
    ),
    'selection_rule': (
        'Calibration and test folds selected by minimizing deviations '
        'from 60/20/20 sizes and class distributions using labels only.'
    ),
    'selected_folds': selected_fold_pairs,
    'partitions': {
        'train': {
            'target_proportion': 0.60,
            'allowed_uses': [
                'model fitting',
                'hyperparameter search by internal cross-validation',
                'preprocessing fitting',
            ],
        },
        'calibration': {
            'target_proportion': 0.20,
            'allowed_uses': [
                'probability calibration',
                'conformal calibration',
                'selection of fixed rejection thresholds defined in protocol',
            ],
            'forbidden_uses': [
                'model fitting',
                'final performance reporting as test evidence',
            ],
        },
        'test': {
            'target_proportion': 0.20,
            'allowed_uses': [
                'single final evaluation after protocol freezing',
            ],
            'forbidden_uses': [
                'feature selection',
                'representation selection',
                'hyperparameter tuning',
                'threshold tuning',
                'calibration fitting',
            ],
        },
    },
    'primary_comparison_factor': 'morphometric representation',
    'representations': REPRESENTATIONS,
    'algorithms_as_sensitivity_factor': [
        'multinomial_logistic_regression',
        'rbf_svm',
        'extra_trees',
        'catboost',
    ],
    'primary_metric': 'macro_f1',
    'secondary_metrics': [
        'balanced_accuracy',
        'matthews_correlation_coefficient',
        'multiclass_log_loss',
        'multiclass_brier_score',
        'expected_calibration_error',
        'area_under_risk_coverage_curve',
        'conformal_marginal_coverage',
        'class_conditional_coverage',
        'mean_prediction_set_size',
    ],
    'hyperparameter_selection': (
        'Five-fold stratified group-aware cross-validation restricted '
        'to the training partition.'
    ),
    'multiplicity_control': 'Holm correction within each prespecified family',
    'bootstrap_replicates_final': 10000,
    'synthetic_scale_factors': [
        0.50, 0.67, 0.80, 1.00, 1.25, 1.50, 2.00
    ],
    'hardware': {
        'recommended': 'Google Colab CPU',
        'gpu_required': False,
    },
    'restrictions': [
        'No claim of real camera-domain validation.',
        'No conversion from pixels to physical units.',
        'No test-set-guided model or representation selection.',
        'Exact duplicate groups must remain within one partition.',
    ],
}

protocol_path = (
    PROTOCOL_DIR
    / 'frozen_split_and_modeling_protocol_v1.json'
)
protocol_path.write_text(
    json.dumps(
        modeling_protocol,
        indent=2,
        ensure_ascii=False,
    ),
    encoding='utf-8',
)

print(json.dumps(modeling_protocol, indent=2, ensure_ascii=False))


## 13. Crear manifiesto de integridad

In [ ]:

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


integrity_rows = []

for path in OUTPUT_DIR.rglob('*'):
    if path.is_file():
        integrity_rows.append({
            'relative_path': str(path.relative_to(OUTPUT_DIR)),
            'size_bytes': path.stat().st_size,
            'sha256': sha256_file(path),
        })

integrity_manifest = pd.DataFrame(
    integrity_rows
).sort_values('relative_path')

integrity_manifest.to_csv(
    RESULTS_DIR / '05_output_integrity_manifest.csv',
    index=False,
    encoding='utf-8-sig',
)

display(integrity_manifest.head(20))


## 14. Validación final

In [ ]:
final_checks = {
    'previous_notebook_validated': True,
    'all_representations_aligned': bool(
        alignment_audit['alignment_exact'].all()
    ),
    'no_group_leakage': bool(
        (
            leakage_audit[
                'groups_in_multiple_partitions'
            ] == 0
        ).all()
    ),
    'no_record_duplication': bool(
        (
            leakage_audit['duplicate_record_ids'] == 0
        ).all()
    ),
    'no_full_row_fingerprint_overlap': bool(
        (
            leakage_audit[
                [
                    'train_calibration_fingerprint_overlap',
                    'train_test_fingerprint_overlap',
                    'calibration_test_fingerprint_overlap',
                ]
            ].to_numpy() == 0
        ).all()
    ),
    'all_datasets_have_three_partitions': bool(
        (
            split_summary.groupby('dataset_id')[
                'partition'
            ].nunique() == 3
        ).all()
    ),
    'all_classes_present_in_all_partitions': bool(
        class_summary.groupby(['dataset_id', 'partition'])[
            'class_label'
        ].nunique().reset_index().apply(
            lambda row: row['class_label'] == (
                7 if row['dataset_id'] == 'dry_bean' else 3
            ), axis=1
        ).all()
    ),
}

final_validation = pd.DataFrame([
    {
        'check': check,
        'passed': passed,
    }
    for check, passed in final_checks.items()
])

final_validation.to_csv(
    RESULTS_DIR / '06_final_split_validation.csv',
    index=False,
    encoding='utf-8-sig',
)

if not final_validation['passed'].all():
    raise RuntimeError(
        'Falló al menos una validación final.'
    )

print('Validación final de particiones: CORRECTA')
display(final_validation)

## 15. Crear ZIP y copiar a Google Drive

In [ ]:

zip_local = (
    LOCAL_ROOT
    / 'NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip'
)

if zip_local.exists():
    zip_local.unlink()

with zipfile.ZipFile(
    zip_local,
    'w',
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(OUTPUT_DIR),
            )


def copy_with_retry(source, destination, attempts=8):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            destination.parent.mkdir(
                parents=True,
                exist_ok=True,
            )
            shutil.copy2(source, destination)

            if (
                not destination.exists()
                or destination.stat().st_size
                != source.stat().st_size
            ):
                raise IOError(
                    'La copia no pudo verificarse.'
                )

            return True

        except (
            ConnectionAbortedError,
            ConnectionResetError,
            OSError,
            IOError,
        ) as error:
            last_error = error
            print(
                f'Intento {attempt}/{attempts}: '
                f'{type(error).__name__}: {error}'
            )
            time.sleep(5)

    print(
        'No se pudo copiar a Drive. '
        f'Último error: {last_error}'
    )
    return False


zip_drive = (
    DRIVE_PROJECT_ROOT / '07_results'
    / 'NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip'
)

copied = copy_with_retry(
    zip_local,
    zip_drive,
)

print('Notebook 03 completado correctamente.')
print('ZIP local:', zip_local)
print('ZIP en Drive:', zip_drive)
print('Copia en Drive verificada:', copied)

if not copied:
    print(
        'Descarga manualmente el ZIP local desde el panel Archivos de Colab.'
    )



## Archivo que debes compartir

Al finalizar, envía:

```text
/content/drive/MyDrive/Grain_Robustness_Q1/07_results/
NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip
```

Si Drive falla, descarga:

```text
/content/Grain_Robustness_Q1_notebook03/
NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip
```

El notebook debe mostrar:

```text
Salida del Notebook 02 validada correctamente.
Alineación de particiones: CORRECTA
Validación final de particiones: CORRECTA
Notebook 03 completado correctamente.
```
